# House Price Prediction — Model Training Notebook

Dataset: **House Price** by Juhi Bhojani — https://www.kaggle.com/datasets/juhibhojani/house-price

This notebook loads the raw dataset, cleans it, explores it, trains and compares regression models, evaluates them, and exports the winning model as `house_price.pkl` for the FastAPI backend.

## 2.1 Load & Inspect

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/house_prices.csv")
print("Shape:", df.shape)
df.head()

Shape: (187531, 21)


,Index,Title,Description,Amount(in rupees),Price (in rupees),location,Carpet Area,Status,Floor,Transaction,...,facing,overlooking,Society,Bathroom,Balcony,Car Parking,Ownership,Super Area,Dimensions,Plot Area
0,0,1 BHK Ready to Occupy Flat for sale in Srushti...,"Bhiwandi, Thane has an attractive 1 BHK Flat f...",42 Lac,6000.0,thane,500 sqft,Ready to Move,10 out of 11,Resale,...,NaN,NaN,Srushti Siddhi Mangal Murti Complex,1,2,NaN,NaN,NaN,NaN,NaN
1,1,2 BHK Ready to Occupy Flat for sale in Dosti V...,One can find this stunning 2 BHK flat for sale...,98 Lac,13799.0,thane,473 sqft,Ready to Move,3 out of 22,Resale,...,East,Garden/Park,Dosti Vihar,2,NaN,1 Open,Freehold,NaN,NaN,NaN
2,2,2 BHK Ready to Occupy Flat for sale in Sunrise...,Up for immediate sale is a 2 BHK apartment in ...,1.40 Cr,17500.0,thane,779 sqft,Ready to Move,10 out of 29,Resale,...,East,Garden/Park,Sunrise by Kalpataru,2,NaN,1 Covered,Freehold,NaN,NaN,NaN
3,3,1 BHK Ready to Occupy Flat for sale Kasheli,This beautiful 1 BHK Flat is available for sal...,25 Lac,NaN,thane,530 sqft,Ready to Move,1 out of 3,Resale,...,NaN,NaN,NaN,1,1,NaN,NaN,NaN,NaN,NaN
4,4,2 BHK Ready to Occupy Flat for sale in TenX Ha...,"This lovely 2 BHK Flat in Pokhran Road, Thane ...",1.60 Cr,18824.0,thane,635 sqft,Ready to Move,20 out of 42,Resale,...,West,"Garden/Park, Main Road",TenX Habitat Raymond Realty,2,NaN,1 Covered,Co-operative Society,NaN,NaN,NaN


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 187531 entries, 0 to 187530
Data columns (total 21 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Index              187531 non-null  int64  
 1   Title              187531 non-null  str    
 2   Description        184508 non-null  str    
 3   Amount(in rupees)  187531 non-null  str    
 4   Price (in rupees)  169866 non-null  float64
 5   location           187531 non-null  str    
 6   Carpet Area        106858 non-null  str    
 7   Status             186916 non-null  str    
 8   Floor              180454 non-null  str    
 9   Transaction        187448 non-null  str    
 10  Furnishing         184634 non-null  str    
 11  facing             117298 non-null  str    
 12  overlooking        106095 non-null  str    
 13  Society            77853 non-null   str    
 14  Bathroom           186703 non-null  str    
 15  Balcony            138596 non-null  str    
 16  Car Parking  

In [3]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Index,187531.0,NaN,NaN,NaN,93765.0,54135.681003,0.0,46882.5,93765.0,140647.5,187530.0
Title,187531,32446,2 BHK Ready to Occupy Flat for sale in Divyasr...,2106,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Description,184508,65634,Multistorey apartment is available for sale. I...,2732,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Amount(in rupees),187531,1561,Call for Price,9684,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Price (in rupees),169866.0,NaN,NaN,NaN,7583.771885,27241.705819,0.0,4297.0,6034.0,9450.0,6700000.0
location,187531,81,new-delhi,27599,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Carpet Area,106858,2758,1000 sqft,5285,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Status,186916,1,Ready to Move,186916,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Floor,180454,947,2 out of 4,12433,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Transaction,187448,4,Resale,144172,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
missing = df.isna().mean().sort_values(ascending=False)
missing

Plot Area            1.000000
Dimensions           1.000000
Society              0.584853
Super Area           0.574225
Car Parking          0.551146
overlooking          0.434254
Carpet Area          0.430185
facing               0.374514
Ownership            0.349366
Balcony              0.260944
Price (in rupees)    0.094198
Floor                0.037738
Description          0.016120
Furnishing           0.015448
Bathroom             0.004415
Status               0.003279
Transaction          0.000443
Amount(in rupees)    0.000000
Title                0.000000
Index                0.000000
location             0.000000
dtype: float64

**Notes:** The dataset has thousands of rows scraped from real listings. Numeric-looking columns such as price, area and floor are stored as free text (e.g. `'42 Lac'`, `'1200 sqft'`, `'3 out of 10'`) and must be parsed. Columns with the most missing values will be handled explicitly in section 2.3 (imputed, dropped, or grouped).

## 2.2 Exploratory Data Analysis (EDA)

In [5]:
import matplotlib
matplotlib.use("Agg")  # safe for headless notebook execution
import matplotlib.pyplot as plt
import seaborn as sns

def parse_amount(x):
    if not isinstance(x, str):
        return None
    x = x.strip().lower()
    try:
        if "lac" in x:
            return float(x.replace("lac", "").strip()) * 1e5
        if "cr" in x:
            return float(x.replace("cr", "").strip()) * 1e7
        return float(x.replace(",", ""))
    except ValueError:
        return None

df["price_clean"] = df["Amount(in rupees)"].apply(parse_amount)
df_plot = df.dropna(subset=["price_clean"])

plt.figure(figsize=(6,4))
sns.histplot(df_plot["price_clean"], log_scale=True)
plt.title("Price distribution (log scale)")
plt.xlabel("Price (INR, log scale)")
plt.savefig("eda_price_distribution.png", bbox_inches="tight")
plt.show()

C:\Users\5g\AppData\Local\Temp\ipykernel_3476\1924475224.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
def parse_area(x):
    if not isinstance(x, str):
        return None
    x = x.strip().lower()
    try:
        if "sqm" in x:
            return float(x.replace("sqm", "").strip()) * 10.764
        if "sqft" in x:
            return float(x.replace("sqft", "").strip())
        return float(x)
    except ValueError:
        return None

df_plot["carpet_area_sqft"] = df_plot["Carpet Area"].apply(parse_area)

plt.figure(figsize=(6,4))
sns.scatterplot(x="carpet_area_sqft", y="price_clean", data=df_plot, alpha=0.4)
plt.yscale("log")
plt.title("Price vs Carpet Area")
plt.savefig("eda_price_vs_area.png", bbox_inches="tight")
plt.show()

C:\Users\5g\AppData\Local\Temp\ipykernel_3476\1166596275.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
top15 = df_plot.groupby("location")["price_clean"].mean().sort_values(ascending=False).head(15)

plt.figure(figsize=(8,5))
top15.plot(kind="bar")
plt.title("Average price by top-15 locations")
plt.ylabel("Average price (INR)")
plt.xticks(rotation=75, ha="right")
plt.savefig("eda_price_by_location.png", bbox_inches="tight")
plt.show()

C:\Users\5g\AppData\Local\Temp\ipykernel_3476\716905115.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
plt.figure(figsize=(6,4))
sns.boxplot(x="Furnishing", y="price_clean", data=df_plot)
plt.yscale("log")
plt.title("Price by furnishing status")
plt.savefig("eda_price_by_furnishing.png", bbox_inches="tight")
plt.show()

C:\Users\5g\AppData\Local\Temp\ipykernel_3476\554009447.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Comments:** Price is heavily right-skewed, which is why we view it on a log scale and later train on `log1p(price)`. Larger carpet area is generally associated with higher price, though there is a lot of noise from location and property type. Average price varies a lot across the top locations, confirming that location is an important feature. Furnished properties tend to trend toward higher prices than unfurnished ones.

## 2.3 Cleaning & Feature Engineering

In [9]:
df["price_clean"] = df["Amount(in rupees)"].apply(parse_amount)
df = df.dropna(subset=["price_clean"])
print("Rows after price cleaning:", df.shape[0])

Rows after price cleaning: 177847


In [10]:
df["carpet_area_sqft"] = df["Carpet Area"].apply(parse_area)
df["super_area_sqft"] = df["Super Area"].apply(parse_area)
# fall back to Super Area when Carpet Area is missing
df["carpet_area_sqft"] = df["carpet_area_sqft"].fillna(df["super_area_sqft"])
df = df.dropna(subset=["carpet_area_sqft"])
print("Rows after area cleaning:", df.shape[0])

Rows after area cleaning: 169577


In [11]:
def parse_floor(x):
    if not isinstance(x, str):
        return None
    x = x.strip().lower()
    try:
        first = x.split("out of")[0].strip()
        if first == "ground":
            return 0
        if first == "basement":
            return -1
        return float(first)
    except (ValueError, IndexError):
        return None

df["floor_num"] = df["Floor"].apply(parse_floor)

In [12]:
for col in ["Bathroom", "Balcony", "Car Parking"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["bathroom"] = df["Bathroom"].fillna(df["Bathroom"].median())
df["balcony"] = df["Balcony"].fillna(0)
df["car_parking"] = df["Car Parking"].fillna(0)

In [13]:
TOP_N = 50
top_locations = df["location"].value_counts().head(TOP_N).index
df["location_grouped"] = df["location"].where(df["location"].isin(top_locations), "other")
df["location_grouped"].value_counts().head()

location_grouped
bangalore    23257
kolkata      21604
new-delhi    21218
gurgaon      18695
hyderabad    11123
Name: count, dtype: int64

In [14]:
drop_cols = ["Index", "Title", "Description", "Dimensions", "Plot Area",
             "Price (in rupees)", "Society", "Amount(in rupees)", "Carpet Area",
             "Super Area", "Floor", "Bathroom", "Balcony", "Car Parking", "location",
             "super_area_sqft", "overlooking", "Status"]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])
df.columns.tolist()

['Transaction',
 'Furnishing',
 'facing',
 'Ownership',
 'price_clean',
 'carpet_area_sqft',
 'floor_num',
 'bathroom',
 'balcony',
 'car_parking',
 'location_grouped']

In [15]:
# Remove outliers based on price-per-sqft
df["price_per_sqft"] = df["price_clean"] / df["carpet_area_sqft"]
low, high = df["price_per_sqft"].quantile([0.01, 0.99])
before = df.shape[0]
df = df[(df["price_per_sqft"] >= low) & (df["price_per_sqft"] <= high)]
df = df.drop(columns=["price_per_sqft"])
print(f"Removed {before - df.shape[0]} outlier rows, {df.shape[0]} rows remain")

Removed 3372 outlier rows, 166205 rows remain


## 2.4 Build a Pipeline & Train

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression

numeric_features = ["carpet_area_sqft", "floor_num", "bathroom", "balcony", "car_parking"]
categorical_features = ["location_grouped", "Furnishing", "Transaction", "Ownership", "facing"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                       ("scale", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                       ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
])

X = df[numeric_features + categorical_features]
y_log = np.log1p(df["price_clean"])

X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

(132964, 10) (33241, 10)


In [17]:
# Note: max_depth / min_samples_leaf keep the exported .pkl file small
# (an unbounded RandomForest trained on ~180k rows can produce a 500MB+ file,
# which is too large to push to GitHub).
models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
    ),
    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=100,
        max_depth=4,
        random_state=42,
    ),
}

pipelines = {}
for name, reg in models.items():
    pipe = Pipeline([("prep", preprocessor), ("reg", reg)])
    pipe.fit(X_train, y_train)
    pipelines[name] = pipe
    print(f"Trained {name}")


Trained LinearRegression
Trained RandomForest
Trained GradientBoosting


## 2.5 Evaluate

In [18]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

results = []
for name, pipe in pipelines.items():
    pred_log = pipe.predict(X_test)
    pred = np.expm1(pred_log)
    actual = np.expm1(y_test)
    mae = mean_absolute_error(actual, pred)
    rmse = root_mean_squared_error(actual, pred)
    r2 = r2_score(actual, pred)
    results.append({"model": name, "MAE": mae, "RMSE": rmse, "R2": r2})

results_df = pd.DataFrame(results).sort_values("R2", ascending=False)
results_df

,model,MAE,RMSE,R2
1,RandomForest,1.203633e+06,4.186790e+06,0.882159
2,GradientBoosting,2.449682e+06,5.186589e+06,0.819158
0,LinearRegression,4.712899e+06,9.493679e+07,-59.590433


In [19]:
best_name = results_df.iloc[0]["model"]
best_model = pipelines[best_name]
print("Winning model:", best_name)

pred_log = best_model.predict(X_test)
pred = np.expm1(pred_log)
actual = np.expm1(y_test)

plt.figure(figsize=(6,6))
plt.scatter(actual, pred, alpha=0.4)
lims = [0, actual.max()]
plt.plot(lims, lims, "r--")
plt.xlabel("Actual price")
plt.ylabel("Predicted price")
plt.title(f"Predicted vs Actual — {best_name}")
plt.savefig("eda_predicted_vs_actual.png", bbox_inches="tight")
plt.show()

Winning model: RandomForest


C:\Users\5g\AppData\Local\Temp\ipykernel_3476\2966224269.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Conclusion:** We trained a Linear Regression baseline plus RandomForest and GradientBoosting tree ensembles, all on `log1p(price)` (inverted with `expm1` at prediction time), which noticeably improved errors versus training directly on raw price. The model with the highest R² and lowest MAE/RMSE on the held-out test set (printed above) is selected as the final model.

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(best_model, X, y_log, cv=5, scoring="r2")
print("5-fold CV R2 scores:", cv_scores)
print("Mean CV R2:", cv_scores.mean())

## 2.6 Export the Model

In [ ]:
import joblib
import json

joblib.dump(best_model, "house_price.pkl")

# Sanity check: reload and predict one sample
loaded = joblib.load("house_price.pkl")
sample = X_test.iloc[[0]]
pred_price = np.expm1(loaded.predict(sample))[0]
print("Reloaded prediction (INR):", pred_price)

json.dump(sorted(df["location_grouped"].unique().tolist()), open("locations.json", "w"))

metrics_out = results_df[results_df["model"] == best_name].to_dict(orient="records")[0]
json.dump(metrics_out, open("metrics.json", "w"))
print("Saved house_price.pkl, locations.json, metrics.json")

Reloaded prediction (INR): 25001829.53776236
Saved house_price.pkl, locations.json, metrics.json


In [ ]:
import sklearn
print("scikit-learn version:", sklearn.__version__)
print("Pin this exact version in backend/requirements.txt")

scikit-learn version: 1.9.0
Pin this exact version in backend/requirements.txt
